In [1]:
import numpy as np
from scipy.ndimage import gaussian_filter
import numpy as np
from scipy.ndimage import gaussian_filter

def kl(p, q):
    """KL divergence where p * log(p/q) is computed only when p != 0"""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    return np.sum(np.where(p != 0, p * np.log(p / (q + 1e-8)), 0))

def smoothed_hist_kl_distance(a, b, nbins = 50):
    """
    Computes symmetric KL divergence between smoothed histograms of two 1D vectors.
    """
    #KL divergence computation
    kl_ab = kl(a, b)
    kl_ba = kl(b, a)

    return kl_ab, kl_ba
    
def compute_kl_bootstrap(class_probs , domain_labels, num_domains=2, repeat=500, nbins=50 ,samples_per_domain=None):
    """
    Computes KL divergence between average class distributions of ROIs grouped by domain.
    """    
    assert class_probs.shape[0] == domain_labels.shape[0], "Mismatch in ROI count"
    kl_values=[] 
    # Pre-split domains
    domain_0 = class_probs[domain_labels == 0]
    domain_1 = class_probs[domain_labels == 1]

    # Balanced sampling size
    max_sample_size = min(len(domain_0), len(domain_1))

    if max_sample_size == 0:
        print("One of the domains has no samples — skipping KL computation.")
        return 0.0, 0.0

    for _ in range(repeat):
        # Random sampling per domain
        probs_bias1 = domain_0[np.random.choice(len(domain_0), max_sample_size, replace=False)]
        probs_bias2 = domain_1[np.random.choice(len(domain_1), max_sample_size, replace=False)]
        
        # print(f"Samples - Bias1: {len(probs_bias1)}, Bias2: {len(probs_bias2)}, Unbias: {len(probs_unbias)}")

        if len(probs_bias1) == 0 or len(probs_bias2) == 0 :
            print("empty group detected")
            continue  # Skip iteration if any group is empty
    
        # Compute average class distribution per domain
        P_bias1 = probs_bias1.mean(axis=0)
        P_bias2 = probs_bias2.mean(axis=0)
        
        #normalize 
        P_bias1 /= P_bias1.sum() + 1e-6
        P_bias2 /= P_bias2.sum() + 1e-6
        
        
        # print(f"P_bias1 = {P_bias1} , P_bias2 = {P_bias2} , P_unbias = {P_unbias}")
       
        # Compute KLs (symmetric: P || Q + Q || P)
        kl_1_2, kl_2_1 = smoothed_hist_kl_distance(P_bias1, P_bias2, nbins)
        
        total_kl = kl_1_2 + kl_2_1
        kl_values.append(total_kl)
    
    return np.median(kl_values), np.std(kl_values)

# > 0.5	Strong domain bias
# 0.1 – 0.5	Moderate domain bias
# 0.01 – 0.1	Weak domain bias (mild imbalance)
# < 0.01	Very good domain invariance (ideal)

In [2]:
import torch
import torch.nn.functional as F

def compute_kl_loss(class_probs, domain_labels, num_domains=2):
    """
    Computes KL loss between average class distributions of domains.
    
    Args:
    
        class_probs: list of Tensors or single Tensor of shape (N, num_classes).
        domain_labels: list of Tensors or single Tensor of shape (N,).
    Returns:
        Scalar KL divergence loss between domain class distributions.
    """
    
    kl_loss = torch.tensor(0.0, device=class_probs.device)
    domain_means = []

    for i in range(num_domains):
        domain_mask = (domain_labels == i)
        if domain_mask.sum() == 0:
            continue
        domain_mean = class_probs[domain_mask].mean(dim=0)
        domain_mean = domain_mean / (domain_mean.sum() + 1e-6)  # Normalize
        domain_means.append(domain_mean)
    num_pairs = len(domain_means) * (len(domain_means) - 1) / 2
    for i in range(len(domain_means)):
        for j in range(i + 1, len(domain_means)):
            pi, pj = domain_means[i], domain_means[j]
            kl_ij = F.kl_div(torch.log(pi + 1e-8), pj, reduction='sum')
            kl_ji = F.kl_div(torch.log(pj + 1e-8), pi, reduction='sum')
            kl_loss += (kl_ij + kl_ji).to(kl_loss.device)         
    kl_loss = kl_loss / num_pairs if num_pairs > 0 else kl_loss

    return kl_loss

In [ ]:
!pip install -U albumentations

In [3]:
def evaluate_kldivergence_testing(epoch,epochs):
    kl_divergenceseval = []
    all_class_probs = []
    all_domain_targets = []
    model.eval()
    # Collect all GT and predictions for the entire epoch
    all_gt_boxes, all_gt_labels = [], []
    all_pred_boxes, all_pred_scores, all_pred_labels = [], [], []
    all_class_probs_tensor=[]
    all_domain_targets_tensor=[]
    with torch.no_grad():
        for images, targets , domain_labels in tqdm(test_loader , desc= f" KL value {epoch+1}/{epochs} " ):
            images = list(img.to(device) for img in images)
            domain_labels = domain_labels.to(device)
            detections , class_logits , domain_targets ,_ ,_  = model(images , targets ,domain_labels)
            class_probs = F.softmax(class_logits, dim=1).detach().cpu()
            domain_targets = domain_targets.detach().cpu()
            all_class_probs.append(class_probs)       # Leave as tensors
            all_domain_targets.append(domain_targets) # Leave as tensors
    
        # Now concatenate properly
        all_class_probs_tensor = torch.cat(all_class_probs, dim=0)         # [N_rois, C]
        all_domain_targets_tensor = torch.cat(all_domain_targets, dim=0)   # [N_rois]
    
        # Filter invalid
        valid_mask = all_class_probs_tensor.sum(dim=1) > 0
        all_class_probs_tensor = all_class_probs_tensor[valid_mask]
        all_domain_targets_tensor = all_domain_targets_tensor[valid_mask]
    
        # Convert to NumPy for downstream KL divergence computation
        all_class_probs = all_class_probs_tensor.numpy()
        all_domain_targets = all_domain_targets_tensor.numpy()
        

        # Compute KL divergence for this epoch
        kl_value, kl_std = compute_kl_bootstrap(all_class_probs , all_domain_targets ,num_domains=2 ,samples_per_domain=1000)
        kl_divergenceseval.append(kl_value)
        print(f"kl_val={max(0.00,kl_value)}")
        print(
            f"KL Divergence: {kl_value:.4f} ± {kl_std:.4f}"
        ) 
    return kl_value
# evaluate_kldivergence_testing()



In [ ]:
#Training
import os
import torch
import albumentations as A
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models.detection as detection
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import numpy as np
import cv2
import torchvision
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.image_list import ImageList
from collections import OrderedDict
from tqdm import tqdm
import pandas as pd
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# === PATHS ===
# chk= "/kaggle/input/debiasednetworkwithseparateclassfierepoch1-26/pytorch/default/1/adversarialnetwork_checkpoint (10).pth"
# metrics="/kaggle/input/debiasednetworkwithseparateclassfierepoch1-26/pytorch/default/1/training_metricsfinal (11).csv"
CHECKPOINT_PATH="E:/SRC-Bhuvaneswari/Debiasing"
METRICS_FILE="E:/SRC-Bhuvaneswari/Debiasing/metrics/Metrics_save.csv"

def custom_collate(batch):
    images = []
    targets = []
    domains = []

    for img, target, domain in batch:
        images.append(img)

        # Append the dictionary directly without converting it to tensor
        targets.append({
            'boxes': target['boxes'],  # Variable-length bounding boxes
            'labels': target['labels']  # Labels
        })
        
        domains.append(domain)

    return images, targets, torch.tensor(domains)


def parse_domain_labels(label_file):
    domain_map = {}
    with open(label_file, 'r') as f:
        for line in f:
            img_path, domain = line.strip().rsplit(' ',1)
            domain_map[img_path] = domain
    return domain_map

def get_label_map(annotations_path):
        label_set = set()
        for file in os.listdir(annotations_path):
            if file.endswith(".xml"):
                file_path = os.path.join(annotations_path, file)
                tree = ET.parse(file_path)
                root = tree.getroot()
                for obj in root.findall("object"):
                    label_set.add(obj.find("name").text)
        label_map = {label: idx + 1 for idx, label in enumerate(sorted(label_set))}
        label_map["background"] = 0
        return label_map

    # return torch.as_tensor(boxes, dtype=torch.float32), torch.as_tensor(labels, dtype=torch.int64)

import xml.etree.ElementTree as ET

def parse_voc_xml(xml_file , label_map):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    boxes = []
    labels = []

    width = int(root.find('size/width').text)
    height = int(root.find('size/height').text)

    for obj in root.findall('object'):
        name = obj.find("name").text
        labels.append(int(label_map[name])+1) # Replace with actual label mapping if needed
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        # print(xmin,ymin,xmax,ymax)
        if xmin >= xmax or ymin >= ymax:
            raise ValueError(f"Invalid bounding box in {file_path}: [{xmin}, {ymin}, {xmax}, {ymax}]")

        boxes.append([xmin, ymin, xmax, ymax])
    return boxes, labels, width, height

class VOCDataset(Dataset):
    def __init__(self, base_day_dir, base_night_dir, transform=None):
        self.data = []
        self.transform = transform

        # Process both day and night
        self._load_voc_folder(base_day_dir, domain_label=0)
        self._load_voc_folder(base_night_dir, domain_label=1)

    def _load_voc_folder(self, base_dir, domain_label):
        image_dir = os.path.join(base_dir, 'JPEGImages')
        anno_dir = os.path.join(base_dir, 'Annotations')
        label_map = get_label_map(anno_dir)
        img_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        
        for idx, img_file in enumerate(img_files):
            img_path = os.path.join(image_dir, img_file)
            xml_path = os.path.join(anno_dir, os.path.splitext(img_file)[0] + '.xml')

            if not os.path.exists(xml_path):
                continue
            
            boxes, labels, width, height = parse_voc_xml(xml_path , label_map)

            if not boxes:
                continue

            # # Rename with suffix
            # suffixed_img_name = f"{prefix}_{idx+1}.jpg"

            self.data.append({
                'image_path': img_path,
                'boxes': boxes,
                'labels': labels,
                'domain': domain_label,
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img = cv2.imread(sample['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        boxes = sample['boxes']
        labels = sample['labels']
        domain = sample['domain']
    
        # Clip boxes to be within image bounds before transformation
        h, w, _ = img.shape
        clipped_boxes = []
        clipped_labels = []
        for box, label in zip(boxes, labels):
            xmin, ymin, xmax, ymax = box
            xmin = max(0, min(xmin, w - 1))
            xmax = max(0, min(xmax, w - 1))
            ymin = max(0, min(ymin, h - 1))
            ymax = max(0, min(ymax, h - 1))
            if xmax > xmin and ymax > ymin:
                clipped_boxes.append([xmin, ymin, xmax, ymax])
                clipped_labels.append(label)
    
        if self.transform:
            transformed = self.transform(image=img, bboxes=clipped_boxes, labels=clipped_labels)
            img = transformed['image']
    
            # Clip transformed boxes again to handle Resize edge errors
            final_boxes = []
            for box in transformed['bboxes']:
                x1, y1, x2, y2 = box
                x1 = max(0, min(x1, 511))
                x2 = max(0, min(x2, 511))
                y1 = max(0, min(y1, 511))
                y2 = max(0, min(y2, 511))
                if x2 > x1 and y2 > y1:
                    final_boxes.append([x1, y1, x2, y2])
    
            target = {
                'boxes': torch.tensor(final_boxes, dtype=torch.float32),
                'labels': torch.tensor(transformed['labels'], dtype=torch.int64),
            }
        else:
            target = {
                'boxes': torch.tensor(clipped_boxes, dtype=torch.float32),
                'labels': torch.tensor(clipped_labels, dtype=torch.int64),
            }
    
        return img, target, torch.tensor(domain)

transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'],check_each_transform=False))

voc_dataset = VOCDataset(base_day_dir='E:/SRC-Bhuvaneswari/Debiasing/voc_day', base_night_dir='E:/SRC-Bhuvaneswari/Debiasing/voc_night', transform=transform)

train_size = int(0.8 * len(voc_dataset))
test_size = len(voc_dataset) - train_size
train_dataset, test_dataset = random_split(voc_dataset, [train_size, test_size])
# train_size = int(0.01 * len(dataset))
# test_size = int(0.01 * len(dataset))
# val = len(dataset) - train_size - test_size
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=custom_collate)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=custom_collate)
print(f"train_images = {train_size} , test_images = {test_size}")

# Gradient Reversal Layer (GRL)
class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class FasterRCNNWithDomain(nn.Module):
    def __init__(self, num_classes, domain_classes):
        super(FasterRCNNWithDomain, self).__init__()

        # Load the pre-trained model
        weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
        self.faster_rcnn = fasterrcnn_resnet50_fpn(weights=weights)
        
        # Modify the classifier to match the number of classes (98 + background = 99)
        in_features = self.faster_rcnn.roi_heads.box_predictor.cls_score.in_features
        self.faster_rcnn.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 98+1)

        # Adversarial bias classifier
        self.domain_classifier_1 = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
        self.domain_classifier_2 = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, images, targets=None,domain_labels=None,alpha=0.1,grl=True):
        device = images[0].device
        
            # ✅ Store original sizes BEFORE transformation
        original_image_sizes = [img.shape[-2:] for img in images]
        # ✅ Use built-in transform for consistency
        images, targets = self.faster_rcnn.transform(images, targets)

        # ✅ Backbone feature extraction
        features = self.faster_rcnn.backbone(images.tensors)
        
        if isinstance(features, torch.Tensor):
            features = OrderedDict([('0', features)])
    
        # ✅ RPN: Generate region proposals
        proposals, proposal_losses = self.faster_rcnn.rpn(images, features, targets)
        # print(f" DEBUG: Number of proposals per image: {[len(p) for p in proposals]}")

        # ✅ RoI Pooling: Extract features for each proposal
        box_features = self.faster_rcnn.roi_heads.box_roi_pool(features, proposals, images.image_sizes)
    
        # ✅ Pass through the box head to get the refined features
        box_features = self.faster_rcnn.roi_heads.box_head(box_features)
    
        # ✅ Get class logits and bounding box regression predictions
        class_logits, bbox_regression = self.faster_rcnn.roi_heads.box_predictor(box_features)

        domain_targets = []
        proposal_counts = [len(p) for p in proposals]
        for i, count in enumerate(proposal_counts):
            domain_label = domain_labels[i].item()  # passed in as external argument
            domain_targets.extend([domain_label] * count)
        
        domain_targets = torch.tensor(domain_targets, device=device)
        
        if self.training:
            # Compute detector losses
            _ , detector_losses = self.faster_rcnn.roi_heads(features, proposals, images.image_sizes, targets)
    
            # Combine RPN and detector losses
            losses = {}
            losses.update(proposal_losses)
            losses.update(detector_losses)



            if grl == True:
                reversed_features = GRL.apply(box_features, alpha)
                
                feat1 = reversed_features[domain_targets == 0]
                feat2 = reversed_features[domain_targets == 1]

                domain_logits1 = self.domain_classifier_1(feat1)

                domain_logits2 = self.domain_classifier_2(feat2) 

            else:
                # Return both detection losses and domain loss output for logging
                feat1 = box_features[domain_targets == 0]
                feat2 = box_features[domain_targets == 1]

                domain_logits1 = self.domain_classifier_1(feat1)

                domain_logits2 = self.domain_classifier_2(feat2)      
                # print("feat1 grl " , feat1)
                # print("feat2 grl " , feat2)
                # print("feat3 grl " , feat3)
            return losses, class_logits, domain_targets ,domain_logits1,domain_logits2
    
        else:
            gt_boxes = [t["boxes"] for t in targets]
            gt_labels = [t["labels"] for t in targets]
            
            gt_boxes = [b.to(device) for b in gt_boxes]
            gt_labels = [l.to(device) for l in gt_labels]
            proposals = [p.to(device) for p in proposals]

            matched_idxs, labels_per_roi = self.faster_rcnn.roi_heads.assign_targets_to_proposals(proposals, gt_boxes, gt_labels)

            pred_labels_per_roi = torch.argmax(class_logits, dim=1)  # Shape: [total_rois]

            # Post-process predictions
            detections = self.faster_rcnn.roi_heads.postprocess_detections(
                class_logits, bbox_regression, proposals, images.image_sizes)
            # Convert tuple to list of dicts
            score_thresh = 0.2 # or make it a class argument for flexibility
            
            formatted_detections = []
            for i in range(len(detections[0])):  
                boxes = detections[0][i]
                scores = detections[1][i]
                labels = detections[2][i]
            
                keep = scores > score_thresh
            
                filtered = {
                    "boxes": boxes[keep],
                    "scores": scores[keep],
                    "labels": labels[keep],
                }
                formatted_detections.append(filtered)
            detections = self.faster_rcnn.transform.postprocess(
                formatted_detections, images.image_sizes, original_image_sizes)                  
            return detections,class_logits, domain_targets ,labels_per_roi ,pred_labels_per_roi

# ==================== METRICS ====================
def iou_score(boxA, boxB):
    """Compute Intersection over Union (IoU) of two boxes: [x1, y1, x2, y2]."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = boxAArea + boxBArea - interArea
    return interArea / unionArea if unionArea > 0 else 0.0

def compute_metrics(
    gt_boxes_list, gt_labels_list,
    pred_boxes_list, pred_scores_list, pred_labels_list,
    iou_threshold=0.5
):
    """
    Compute:
      - Precision, Recall, F1
      - Average IoU over all true-positive matches
      - mAP via area under the global precision-recall curve.

    For each predicted box, we determine if it is a true positive (TP) or false positive (FP).
    Unmatched ground-truth boxes are false negatives (FN).
    """
    TP, FP, FN = 0, 0, 0
    total_iou = 0.0  # Sum of IoUs for all true-positive matches
    iou_count = 0    # How many TPs we have (for averaging IoU)

    # For mAP computation: collect each prediction's score and a binary indicator of match or not
    all_pred_scores_for_map = []
    all_pred_match_for_map = []

    for gt_boxes, gt_labels, pred_boxes, pred_scores, pred_labels in zip(
        gt_boxes_list, gt_labels_list, pred_boxes_list, pred_scores_list, pred_labels_list):
        detected_indices = []  # which GT boxes are matched

        # Loop over predicted boxes
        for i, pred_box in enumerate(pred_boxes):
            pred_score = float(pred_scores[i])
            pred_label = pred_labels[i] if np.isscalar(pred_labels[i]) else pred_labels[i].item()

            # If no GT boxes, it's automatically a false positive
            if len(gt_boxes) == 0:
                FP += 1
                all_pred_scores_for_map.append(pred_score)
                all_pred_match_for_map.append(0)
                continue

            match_found = False
            for j, gt_box in enumerate(gt_boxes):
                gt_label = gt_labels[j] if np.isscalar(gt_labels[j]) else gt_labels[j].item()

                # Check label + IoU + not matched
                if (pred_label == gt_label) and (j not in detected_indices):
                    iou = iou_score(gt_box, pred_box)
                    if iou >= iou_threshold:
                        # True positive
                        TP += 1
                        detected_indices.append(j)
                        match_found = True

                        total_iou += iou
                        iou_count += 1

                        all_pred_scores_for_map.append(pred_score)
                        all_pred_match_for_map.append(1)  # matched
                        break
            if not match_found:
                FP += 1
                all_pred_scores_for_map.append(pred_score)
                all_pred_match_for_map.append(0)

        # Any ground truth box not matched => false negative
        FN += (len(gt_boxes) - len(detected_indices))

    # ---- Precision, Recall, F1 ----
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # ---- Average IoU over all TPs ----
    avg_iou = total_iou / iou_count if iou_count > 0 else 0.0

    # ---- Compute mAP via the global PR curve ----
    all_pred_scores_for_map = np.array(all_pred_scores_for_map)
    all_pred_match_for_map = np.array(all_pred_match_for_map, dtype=int)

    if len(all_pred_scores_for_map) == 0:
        map_value = 0.0
    else:
        precision_curve, recall_curve, _ = precision_recall_curve(
            all_pred_match_for_map, all_pred_scores_for_map
        )
        map_value = auc(recall_curve, precision_curve)
    return precision, recall, f1_score, avg_iou, map_value


    # === TRAINING ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FasterRCNNWithDomain(15 , 3).to(device)
optimizer = torch.optim.SGD([
    {'params': model.faster_rcnn.parameters(), 'lr': 1e-3},  
    {'params': model.domain_classifier_1.parameters(), 'lr': 1e-4},
    {'params': model.domain_classifier_2.parameters(), 'lr': 5e-4}
], momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()
print("training started successfully")
# # Load checkpoint if exists
# CHECKPOINT_PATH="/kaggle/input/debiasednetworkepoch27-30-final/pytorch/default/1/adversarialnetwork_checkpoint (12).pth"
# if os.path.exists(CHECKPOINT_PATH):
#     checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
#     model.load_state_dict(checkpoint["model_state_dict"])
#     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
#     start_epoch = checkpoint["epoch"]
#     CHECKPOINT_PATH="/kaggle/working/adversarialnetwork_checkpoint.pth"
#     METRICS_FILE="/kaggle/working/training_metricsfinal.csv"
#     print("metrics updated as '/kaggle/working/training_metricsfinal.csv'")
#     print(" checkpoint updated as '/kaggle/working/adversarialnetwork_checkpoint.pth' ")
#     print(f"model loaded successfully ")

epochs = 20
# Store KL divergence values over epochs
kl_divergences = []
for epoch in range(epochs):
    model.train()
     # ✅ Ensure the CSV file exists with headers
    if not os.path.exists(METRICS_FILE):
        pd.DataFrame(columns=["Epoch", "total_domain_loss_entrophy","total_object_loss","total_object_loss_with_entrophy_kl","total_domain_loss_grl","total_kl_loss","KL Divergence", "Precision", "Recall", "F1", "Avg_IoU", "mAP"]).to_csv(METRICS_FILE, index=False)
    total_object_loss_with_entrophy, total_domain_loss_grl,total_domain_loss_entrophy ,total_object_loss , total_kl_loss = 0, 0 ,0 ,0,0
    domain_entropy_count = 0
    # Collect all GT and predictions for the entire epoch
    all_gt_boxes, all_gt_labels = [], []
    all_pred_boxes, all_pred_scores, all_pred_labels = [], [], []
    all_class_probs = []
    all_domain_targets = []
    valid_kl_batches = 0
    
    def neg_entropy_loss(logits):
        if logits.shape[0] == 0:
            return None  # or torch.tensor(0.0, device=logits.device) if you prefer
        probs = F.softmax(logits, dim=1)
        return torch.mean(torch.sum(probs * torch.log(probs + 1e-6), dim=1))

        
    def is_valid(tensor):
        return tensor is not None and not torch.isnan(tensor).any() and not torch.isinf(tensor).any()

    for images, targets, domain_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        # print(targets[0].keys())
        domain_labels = domain_labels.to(device)
        optimizer.zero_grad()
        epoch_fraction = epoch / epochs
        frcnn_output, class_logits, domain_targets , domain_logits1 ,domain_logits2 = model(images, targets , domain_labels , alpha = 1.0 if epoch > 5 else 0.1 ,grl= False)
        
        domain_entropy_loss_sum = 0
        count = 0
        
        domainloss1 = neg_entropy_loss(domain_logits1)
        domainloss2 = neg_entropy_loss(domain_logits2)

        # Calculate negative entropy losses for domain classifiers
        if domainloss1 is not None:
            count+=1
            domain_entropy_loss_sum += domainloss1.item() * domain_logits1.shape[0]
            domain_entropy_count += domain_logits1.shape[0]
        if domainloss2 is not None:
            count+=1
            domain_entropy_loss_sum += domainloss2.item() * domain_logits2.shape[0]
            domain_entropy_count += domain_logits2.shape[0]
 

        if domain_entropy_count > 0:
            domain_loss_entrophy = domain_entropy_loss_sum 
        else:
            print(f"there is no entrophy loss in epoch ({epoch})")
            domain_loss_entrophy = 0.0
            
        loss_object = sum(loss for loss in frcnn_output.values())
                
        # Already on GPU: class_logits, domain_targets
        class_probs = F.softmax(class_logits, dim=1)
        domain_targets_batch = domain_targets
        
        # Apply valid mask within this batch
        valid_mask = class_probs.sum(dim=1) > 0
        
        # Filter
        class_probs = class_probs[valid_mask]
        domain_targets_batch = domain_targets_batch[valid_mask]
        
        # Compute per-batch KL divergence
        if class_probs.size(0) > 0:
            kl_loss = compute_kl_loss(class_probs, domain_targets_batch)
        else:
            kl_loss = torch.tensor(0.0, device=class_probs.device)

        if torch.isfinite(kl_loss):  # avoid NaNs/infs
            valid_kl_batches += 1
        kl_lambda = min(1.0, epoch / (0.3 * epochs))
        lambda_val=0.01
        
        object_loss_with_entrophy = loss_object + (lambda_val * domain_loss_entrophy ) + kl_lambda * kl_loss
    
        # print(f"loss_object = {loss_object} , domain_loss = {domain_loss} ")

        object_loss_with_entrophy.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) 
        optimizer.step()
        optimizer.zero_grad()

        frcnn_output, class_logits, domain_targets , domain_logits1, domain_logits2 = model(images, targets,domain_labels,alpha = 1.0 if epoch > 5 else 0.1 ,grl= True)
        
        # Create matching target labels (all zeros, because this classifier should output class 0 for bias_1)
        target1 = torch.zeros(domain_logits1.size(0), dtype=torch.long, device=device)
        target2 = torch.zeros(domain_logits2.size(0), dtype=torch.long, device=device)
        
        # Compute loss
        domainloss1grl = criterion(domain_logits1, target1)
        domainloss2grl = criterion(domain_logits2, target2)

        templossdomaingrl = 0
        # print(f"domainloss1grl = {domainloss1grl} ,domainloss2grl = {domainloss2grl} , domainloss3grl = {domainloss3grl} ")
        if is_valid(domainloss1grl) :
            # print("domainloss1grl is valid")
            templossdomaingrl += domainloss1grl
        if is_valid(domainloss2grl) :
            templossdomaingrl += domainloss2grl
            # print("domainloss2grl is valid")
        loss_grl = templossdomaingrl
        # print(f"kl_loss = {kl_loss} , loss_grl = {loss_grl} ")
        loss_grl.backward()  # Compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) 
        optimizer.step()  # Update model parameters
    
        total_object_loss_with_entrophy += object_loss_with_entrophy.item()
        total_domain_loss_entrophy += domain_loss_entrophy
        total_domain_loss_grl += loss_grl.item()
        total_object_loss+=loss_object.item()
        total_kl_loss+=kl_loss.item()
        # Switch to eval mode to get predictions
        model.eval()
        with torch.no_grad():
            detections, class_logits, domain_targets_eval , _ , _ = model(images , targets ,domain_labels)
        model.train()
        # Accumulate GT & predictions
        for t, o in zip(targets, detections):
            all_gt_boxes.append(t["boxes"].cpu().numpy())
            all_gt_labels.append(t["labels"].cpu().numpy())
            all_pred_boxes.append(o["boxes"].cpu().numpy())
            all_pred_scores.append(o["scores"].cpu().numpy())
            all_pred_labels.append(o["labels"].cpu().numpy())

    # for testing delete it later
    model.eval()
    kl_value = evaluate_kldivergence_testing(epoch,epochs)       
    kl_divergences.append(kl_value)
    # Compute metrics for the entire epoch
    precision, recall, f1_score, avg_iou, map_value = compute_metrics(
        all_gt_boxes,
        all_gt_labels,
        all_pred_boxes,
        all_pred_scores,
        all_pred_labels,
        iou_threshold=0.5)

    print(
        f"Epoch {epoch+1}/{epochs},"
        f"total_domain_loss_entrophy: {total_domain_loss_entrophy / domain_entropy_count:.4f},"
        f"total_object_loss: {total_object_loss/len(train_loader):.4f},"
        f"total_object_loss_with_entrophy_kl: {total_object_loss_with_entrophy/len(train_loader):.4f},"
        f"total_domain_loss_grl: {total_domain_loss_grl },"
        f"total_domain_loss_grl_norm: {total_domain_loss_grl / domain_entropy_count:.4f},"
        #f"KL Divergence: {kl_value:.4f} ± {kl_std:.4f}"
        f"total_kl_loss : {total_kl_loss/valid_kl_batches:.4f},"
        f"Precision: {precision:.4f}, "
        f"Recall: {recall:.4f}, "
        f"F1: {f1_score:.4f}, "
        f"Avg_IoU: {avg_iou:.4f}, "
        f"mAP: {map_value:.4f}" )
    

    # ✅ Save checkpoint
    checkpoint={
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }

    path = f"debiasemodel.pth"
    torch.save(checkpoint, os.path.join(CHECKPOINT_PATH, path))


    # ✅ Save metrics after each epoch (append to CSV)
    new_data = pd.DataFrame([{
        "Epoch": epoch + 1,
        "total_domain_loss_entrophy": total_domain_loss_entrophy/domain_entropy_count,
        "total_object_loss": total_object_loss/len(train_loader),
        "total_object_loss_with_entrophy_kl": total_object_loss_with_entrophy/len(train_loader),
        "total_domain_loss_grl": total_domain_loss_grl/domain_entropy_count,
        "total_kl_loss":total_kl_loss/valid_kl_batches,
        "kl_divergence": kl_value,
        "Precision": precision,
        "Recall": recall,
        "F1": f1_score,
        "Avg_IoU": avg_iou,
        "mAP": map_value,
        
    }])
    torch.cuda.empty_cache()
    new_data.to_csv(METRICS_FILE, mode="a", header=False, index=False)  # Append without rewriting headers
# Plot KL divergence trend
print("Trained succesfully!!!!")
plt.plot(range(epochs), kl_divergences, label="KL Divergence")
plt.xlabel("Epoch")
plt.ylabel("KL Divergence")
plt.legend()
plt.title("KL Divergence Over Training")
plt.show()
print("✅ Training complete!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
Class_Mapping = ['Background','AmurTiger', 'Badger', 'BlackBear', 'Cow', 'Dog', 'Hare', 'Leopard', 'LeopardCat', 'MuskDeer', 'RaccoonDog', 'RedFox', 'RoeDeer', 'Sable', 'SikaDeer', 'Weasel', 'WildBoar', 'Y.T.Marten']

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Visualize 20 images from the test loader
num_images = 20
fig, axes = plt.subplots(num_images, 2, figsize=(16, num_images * 6))

with torch.no_grad():
    for idx, (images, targets , domain_lables) in enumerate(test_loader):
        if idx >= num_images:
            break
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Get model predictions
        outputs, _, _, _, _ = model(images, targets, domain_labels=domain_lables)

        # Get first image and target (assuming batch size 1 for visualization)
        img = images[0].cpu().permute(1, 2, 0).numpy()
        img = (img * 0.229) + 0.485  # Unnormalize
        img = img.clip(0, 1)

        # Plot Ground Truth
        ax_gt = axes[idx, 0]
        ax_gt.imshow(img)
        ax_gt.set_title("Ground Truth")
        for box, label in zip(targets[0]["boxes"].cpu(), targets[0]["labels"].cpu()):
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor="green", facecolor="none")
            ax_gt.add_patch(rect)
            ax_gt.text(xmin, ymin - 5, f"{Class_Mapping[int(label.item())-1]}", color="green", fontsize=15, weight="bold")

        # Plot Predictions
        ax_pred = axes[idx, 1]
        ax_pred.imshow(img)
        ax_pred.set_title("Prediction")
        for box, label, score in zip(outputs[0]["boxes"].cpu(), outputs[0]["labels"].cpu(), outputs[0]["scores"].cpu()):
            if score > 0.5:  # Confidence threshold
                xmin, ymin, xmax, ymax = box
                rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                         linewidth=2, edgecolor="blue", facecolor="none")
                ax_pred.add_patch(rect)
                ax_pred.text(xmin, ymin - 5, f"{Class_Mapping[int(label.item())-1]} ({score:.2f})", color="blue", fontsize=15, weight="bold")

        ax_gt.axis("off")
        ax_pred.axis("off")

plt.tight_layout()
plt.show()



In [ ]:
import torch
import matplotlib.pyplot as plt
import random
import torchvision.transforms as T
import torchvision

# === FUNCTION TO CALCULATE TEST ACCURACY ===
def calculate_test_accuracy(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, targets, domain_labels in tqdm(test_loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            outputs, _, _, _, _ = model(images, targets, domain_labels)
            for t, o in zip(targets, outputs):
                gt_labels = t["labels"].cpu().numpy()
                pred_labels = o["labels"].cpu().numpy()
                total += len(gt_labels)
                correct += sum([1 for gt, pred in zip(gt_labels, pred_labels) if gt == pred])
    accuracy = 100 * correct / total if total > 0 else 0
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

# === TEST ACCURACY CALCULATION ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Calculating test accuracy...")
calculate_test_accuracy(model, test_loader, device)


In [2]:
#BA
import torch

def bog_attribute_to_task(bog_tilde, bog_gt_g, bog_tilde_train=None, toprint=True, disaggregate=False, num_attributes=None, total_images=None, num_attributes_train=None, total_images_train=None):
    device = bog_tilde.device

    if num_attributes is None:  # need to be provided if multi-label
        num_attributes = torch.sum(bog_tilde, dim=0)
    if total_images is None:  # attribute not mutually exclusive
        total_images = torch.sum(num_attributes)
    if bog_tilde_train is None:
        bog_tilde_train = bog_tilde
    if num_attributes_train is None:
        num_attributes_train = torch.sum(bog_tilde_train, dim=0)
    if total_images_train is None:
        total_images_train = torch.sum(num_attributes_train)

    data_bog = bog_tilde / num_attributes.unsqueeze(0)
    pred_bog = bog_gt_g / num_attributes.unsqueeze(0)

    p_t_a = bog_tilde_train / num_attributes_train.unsqueeze(0)
    p_t = torch.sum(bog_tilde_train, dim=1) / total_images_train

    diff = pred_bog - data_bog

    indicator = torch.sign(p_t_a - p_t.unsqueeze(1))  # p_t shape (num_classes,), unsqueeze to (num_classes, 1)
    diff = torch.where(indicator == 0, torch.zeros_like(diff), diff)
    diff = torch.where(indicator == -1, -diff, diff)

    if disaggregate:
        diff_before = diff.clone()
    value = torch.nanmean(diff)

    if toprint:
        print(f"Attribute->Task: {value.item():.6f}")

    if disaggregate:
        return diff_before, value
    return value

def get_at(labels, preds, num_classes=98, num_domains=3):
    device = labels.device

    bog_tilde = torch.zeros((num_classes, num_domains), device=device)
    bog_gt_g = torch.zeros((num_classes, num_domains), device=device)

    tasks = labels[:, 0].long()
    domains = labels[:, 1].long()
    pred_tasks = preds[:, 0].long()

    # Batch update
    bog_gt_g.index_put_((tasks, domains), torch.ones_like(tasks, dtype=bog_gt_g.dtype), accumulate=True)
    bog_tilde.index_put_((pred_tasks, domains), torch.ones_like(pred_tasks, dtype=bog_tilde.dtype), accumulate=True)

    return bog_attribute_to_task(bog_tilde, bog_gt_g, toprint=False)

def bootstrap_bias_amp(domain, targets, pred, repeat=500, num_classes=98, num_domains=3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Make sure everything is torch tensor and moved to device
    if not torch.is_tensor(domain):
        domain = torch.from_numpy(domain).to(device)
    else:
        domain = domain.to(device)

    if not torch.is_tensor(targets):
        targets = torch.from_numpy(targets).to(device)
    else:
        targets = targets.to(device)

    if not torch.is_tensor(pred):
        pred = torch.from_numpy(pred).to(device)
    else:
        pred = pred.to(device)

    test_labels = torch.stack((targets, domain), dim=1)
    test_pred = torch.stack((pred, domain), dim=1)
    max_val = targets.size(0)

    auc_bias = []
    for _ in range(repeat):
        idx = torch.randint(0, max_val, (max_val,), device=device)
        auc_bias.append(get_at(test_labels[idx], test_pred[idx], num_classes=num_classes, num_domains=num_domains))

    auc_bias = torch.tensor(auc_bias, device=device)
    return torch.median(auc_bias).item(), torch.std(auc_bias).item()


In [3]:
#DEO
import numpy as np
from sklearn.metrics import recall_score

# Function to compute DEO
def bootstrap_deo(domain_labels, targets, predictions, repeat=1000):
    """
    Compute DEO for Faster R-CNN object detection.

    Parameters:
        domain_labels (np.array): Domain labels (0, 1, 2) indicating bias groups.
        targets (np.array): Ground truth fish species labels (1 to 98).
        predictions (np.array): Predicted fish species labels (1 to 98).
        repeat (int): Number of bootstrap samples.

    Returns:
        float: Median DEO
        float: Standard deviation of DEO
    """
    
    domain_labels = np.array(domain_labels)
    targets = np.array(targets)
    predictions = np.array(predictions)

    valid_mask = (targets > 0) & (predictions > 0)  # adjust logic if needed
    targets = targets[valid_mask]
    predictions = predictions[valid_mask]
    domain_labels = domain_labels[valid_mask]

    deo = np.zeros(repeat)

    # Get indices for each domain
    idx_0 = np.where(domain_labels == 0)[0]
    idx_1 = np.where(domain_labels == 1)[0]
    idx_2 = np.where(domain_labels == 2)[0]

    # Find balanced sample size across all domains
    min_samples = min(len(idx_0), len(idx_1), len(idx_2))

    if min_samples == 0:
        print("One of the groups has no samples — cannot compute DEO.")
        return 0.0, 0.0

    for i in range(repeat):
        s0 = np.random.choice(idx_0, min_samples, replace=False)
        s1 = np.random.choice(idx_1, min_samples, replace=False)
        s2 = np.random.choice(idx_2, min_samples, replace=False)

        combined_idx = np.concatenate([s0, s1, s2])
        

        domain_i = domain_labels[combined_idx]
        targets_i = targets[combined_idx]
        pred_i = predictions[combined_idx]
    
        g0 = np.where(domain_i == 0)[0]
        g1 = np.where(domain_i == 1)[0]
        g2 = np.where(domain_i == 2)[0]

        # print(f"g0 = {len(g0)} , g1 = {len(g1)} , g2 = {len(g2)} samples ")

        # Compute macro recall for each group
        recall_0 = recall_score(targets_i[g0], pred_i[g0], average="macro", zero_division=1) if len(g0) > 0 else 0
        recall_1 = recall_score(targets_i[g1], pred_i[g1], average="macro", zero_division=1) if len(g1) > 0 else 0
        recall_2 = recall_score(targets_i[g2], pred_i[g2], average="macro", zero_division=1) if len(g2) > 0 else 0

        deo[i] = max(abs(recall_0 - recall_1), abs(recall_1 - recall_2), abs(recall_0 - recall_2))

    return np.median(deo), np.std(deo)


In [ ]:
def evaluate_DEO_testing():
    all_domain_labels=[]
    all_class_labels=[]
    all_pred_labels=[]
    model.eval()
    print(device)
    with torch.no_grad():
        for images, targets , domain_labels in test_loader:
            images = list(img.to(device) for img in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            # targets: list of dicts, one per image in batch
            domain_labels = domain_labels.to(device)
            detections, class_logits, domain_targets, labels_per_roi ,pred_labels_per_roi = model(images , targets ,domain_labels)
            for roi_labels in labels_per_roi:  # list of tensors
                all_class_labels.extend(roi_labels.cpu().tolist())
            domain_targets = domain_targets.detach().cpu().numpy()
            all_domain_labels.extend(domain_targets)
            all_pred_labels.extend(pred_labels_per_roi.cpu().tolist())   

        # print(f"domain_labels: {len(all_domain_labels)}")
        # print(f"class_labels: {len(all_class_labels)}")
        # print(f"pred_labels: {len(all_pred_labels)}")


        all_class_labels = np.array(all_class_labels)
        all_domain_labels = np.array(all_domain_labels) 
        all_pred_labels = np.array(all_pred_labels)

        # #debugging
        # unique_vals, counts = np.unique(all_domain_labels, return_counts=True)
        # print("Domain label distribution:")
        # for val, count in zip(unique_vals, counts):
        #     print(f"Label {val}: {count} samples")

    
    # median_value,stnd_value = bootstrap_deo(all_domain_labels, all_class_labels, all_pred_labels, repeat=500)
    # print(f"DEO Median: {median_value:.4f}, Std: {stnd_value:.4f}")
    
    ba, ba_std = bootstrap_bias_amp(
        domain=all_domain_labels,
        targets=all_class_labels,
        pred=all_pred_labels,
        num_classes=98,
        num_domains=3,
    )
    
    print(f"Bias Amplification Median: {ba:.4f}, Std: {ba_std:.4f}")

   
evaluate_DEO_testing()
# DEO
# 0.00 - 0.05	✅ Good — low disparity across groups
# 0.05 - 0.10	⚠️ Moderate — some disparity; acceptable for many use cases
# > 0.10	    ❌ High — significant fairness gap between groups

# BA
# 0.0 → perfect: predictions match the bias distribution of GT labels exactly.
# ~0.1–0.2 → generally acceptable in balanced setups.
# > 0.3 → increasing amplification of bias.
# > 0.5 → strong domain bias in predictions.